# Member 5 MVP — YU Guanlin

Frozen SBERT sentence vectors and token sequences → independent TextCNN.
Uses the published clean_v2 / folds_v1 and Member 4 training interface. The accompanying evidence records actual script runs. Default mode displays those results; set `RUN_EXPERIMENTS=True` to regenerate caches and retrain both representations.

Before running in Colab, clone the private repository on branch `m5-textcnn-mvp-20261107` using your own authorized access and change into the clone. Install `Member5_file/requirements.txt` in a compatible Python runtime and restart the kernel if core libraries changed. No credential belongs in notebook source or outputs. GPU execution is supported by `--device cuda`; this MVP's measured evidence uses CPU.

The test partition was already inspected by another team experiment. This notebook saves test probabilities but does not evaluate test labels. Do not represent it as a pristine team holdout.

In [1]:
import sys, json, subprocess
from pathlib import Path
import pandas as pd
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/config.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Run this notebook from inside the repository clone')
sys.path.insert(0, str(ROOT / 'Member5_file'))
sys.path.insert(0, str(ROOT))
from mvp import inputs, verify_embeddings, validate_probabilities
from src.cache import load_preds
RUN_EXPERIMENTS = False
DEVICE = 'cpu'
print('Repository:', ROOT)


Repository: /Users/yuguanlin/Library/Mobile Documents/com~apple~CloudDocs/大三上/DSAI4205/Group_project/PolyU_DSAI4205_Group_Project


## Run or reproduce

The full run performs frozen encoding, then 5 outer folds for each TextCNN representation. An inner grouped split selects epochs; each outer model is refitted without outer validation labels. FastText is trained on the corresponding outer training corpus only. Fixed CNN kernels are 2/3/4/5 with 100 filters each and dropout 0.5; the epoch budget is 12 with patience 3. This is a bounded first experiment rather than an exhaustive search.

In [2]:
if RUN_EXPERIMENTS:
    script = str(ROOT / 'Member5_file/mvp.py')
    for args in [['encode'], ['train', '--embedding', 'fasttext'], ['train', '--embedding', 'transformer']]:
        subprocess.run([sys.executable, script, *args, '--device', DEVICE], cwd=ROOT, check=True)
clean, splits = inputs()
print('Clean rows:', len(clean), '| Development:', (splits.split == 'train').sum(), '| Test:', (splits.split == 'test').sum())


Clean rows: 1525 | Development: 1222 | Test: 303


In [3]:
evidence = ROOT / 'Member5_file/evidence'
summaries = [json.loads(p.read_text()) for p in sorted(evidence.glob('textcnn__*/summary.json'))]
if not summaries:
    raise RuntimeError('No completed evidence: run the experiments first')
comparison = pd.DataFrame([{'run_id': s['run_id'], 'embedding': s['metrics']['embedding'],
    'accuracy': s['metrics']['accuracy'], 'macro_f1': s['metrics']['macro_f1'],
    'baseline_macro_f1': s['baseline_macro_f1'], 'delta': s['difference_from_baseline'],
    'fold_sd': s['fold_macro_f1_sample_sd'], 'test_evaluated': s['test_evaluated']} for s in summaries])
display(comparison)


,run_id,embedding,accuracy,macro_f1,baseline_macro_f1,delta,fold_sd,test_evaluated
0,textcnn__fasttext_tok__20261107-141250-6a8e,fasttext_tok,0.53437,0.536124,0.581996,-0.045872,0.056843,False
1,textcnn__transformer_tokens__20261107-141902-c914,transformer_tokens,0.58347,0.579576,0.581996,-0.002420,0.027189,False


## Handoff integrity

Large `.npy`, `.npz` and weight files are local/shared-storage artifacts and are excluded from Git. Evidence tables and manifests are committed. To validate local caches, run the following optional cell after placing shared artifacts under `DATA_ROOT`. Every row is aligned by stable ID, and prediction probability columns are `[Neutral, Positive, Negative]`.

In [4]:
CHECK_LOCAL_CACHES = False
if CHECK_LOCAL_CACHES:
    verify_embeddings(clean)
    dev_ids = set(splits.loc[splits.split == 'train', 'id'])
    test_ids = set(splits.loc[splits.split == 'test', 'id'])
    for s in summaries:
        pred = load_preds(s['run_id'])
        validate_probabilities(pred['oof_ids'], pred['oof_proba'])
        validate_probabilities(pred['test_ids'], pred['test_proba'])
        assert set(pred['oof_ids']) == dev_ids and set(pred['test_ids']) == test_ids
print('Review mode complete. Cache checks enabled:', CHECK_LOCAL_CACHES)


Review mode complete. Cache checks enabled: False
